# 01 清洗：从发票原始数据到价格回归面板

**输入**：`sample_GX1701/02/03.csv`、1701。1702。1703 是 2017 年抽样三分之一。

**输出**（3 个 `.dta`，都落在 `G:\Kuangyu_Temp\Outsource\productivity\`）：
- `invoice_panel.dta`　　主回归面板：firm × product × city × year，含购买价
- `market_conds.dta`　市场条件：product × city × year（买家数、卖家数、总量、均价）
- `firm_chars.dta`　　企业特征：firm × year，从 `full_data.dta` 提取

**设计决策**（已与用户确认）：
1. 项目代码 不足 19 位 → **右补零** → 取前 9 位 → 匹配 bianma，未匹配丢弃
2. 时间粒度：**年**（原始只有 2017）
3. city = **购方地区**
4. 红冲对冲：同 (购方, 销方, product_9, year) 取净；净金额或净数量 ≤ 0 则 drop
5. 企业特征直接从 `full_data.dta` 拿


In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

BASE = r'G:\Kuangyu_Temp\Outsource'
OUT  = os.path.join(BASE, 'productivity')
os.makedirs(OUT, exist_ok=True)

print('BASE =', BASE)
print('OUT  =', OUT)

## Step 1　读入三个 CSV，拼接

`dtype` 里明确把企业 ID、地区代码、项目代码、数量、单位读为字符串，避免长整数/科学计数法问题。数值列（金额、单价、税额）让 pandas 自动推断。

In [ ]:
files = ['sample_GX1701.csv', 'sample_GX1702.csv', 'sample_GX1703.csv']

dtypes = {
    '购方企业ID': str, '销方企业ID': str,
    '购方地区':   str, '销方地区':   str,
    '项目代码':   str, '项目':       str,
    '单位':       str, '数量':       str,
}

parts = []
for f in files:
    p = os.path.join(BASE, f)
    t = pd.read_csv(p,
                    dtype=dtypes,
                    na_values=['', 'NULL', '(Null)', 'null'],
                    encoding='utf-8',
                    parse_dates=['开票日期'])
    t['src'] = f.replace('sample_GX', '').replace('.csv', '')
    print(f, '->', len(t))
    parts.append(t)

df = pd.concat(parts, ignore_index=True)
print('\ntotal rows:', len(df))
print('memory (MB):', df.memory_usage(deep=True).sum() / 1024**2)

In [ ]:
df.head()

In [ ]:
df.dtypes

## Step 2　数值列转换 + 缺失诊断

`数量` 原是文本，需转数值；同时检查三个金额列的缺失。

In [ ]:
df['数量']     = pd.to_numeric(df['数量'], errors='coerce')
df['开票金额'] = pd.to_numeric(df['开票金额'], errors='coerce')
df['单价']     = pd.to_numeric(df['单价'], errors='coerce')
df['税额']     = pd.to_numeric(df['税额'], errors='coerce')

miss = pd.DataFrame({
    'missing_rate': df[['数量','开票金额','单价','税额','项目代码','购方地区','销方地区']].isna().mean(),
})
miss

## Step 3　清洗 `项目代码` → 9 位 product_id

处理顺序：
1. 项目代码缺失或含非数字符 → 丢
2. 代码长度 < 19 → 右补 0
3. 代码长度 > 19 → 取前 19（避免有人多填字符）
4. 取前 9 位 → `product_id`

In [ ]:
raw_n = len(df)

# 仅保留纯数字项目代码
mask_digits = df['项目代码'].fillna('').str.fullmatch(r'\d+')
print('非纯数字/空 项目代码 → drop :', (~mask_digits).sum(),
      f'({(~mask_digits).mean()*100:.2f}%)')
df = df[mask_digits].copy()

# 右补零到 19 位
code = df['项目代码'].str.ljust(19, '0')
# 长于 19 位的（极少）取前 19
df['code19']      = code.str[:19]
df['product_id']  = df['code19'].str[:9]

print('rows kept:', len(df), f'({len(df)/raw_n*100:.2f}% of raw)')
df[['项目代码','code19','product_id']].head()

## Step 4　匹配 `bianma.dta`（只留有效 9 位产品）

In [ ]:
bianma = pd.read_stata(os.path.join(BASE, 'bianma.dta'))
print('bianma shape:', bianma.shape, '\ncols:', list(bianma.columns))

valid_pids = set(bianma['product_id'].astype(str))
print('unique product_id in bianma:', len(valid_pids))

before = len(df)
df = df[df['product_id'].isin(valid_pids)].copy()
print(f'matched: {len(df):,} of {before:,}  ({len(df)/before*100:.2f}%)')
print('unique 9-digit products in invoice data:', df['product_id'].nunique())

## Step 5　添加 year、处理数量缺失、删自交易

- year 从开票日期取
- 数量缺失但单价与金额都在 → 反推：数量 = (金额 - 税额)/单价（单价是含税前单价）
- 仍缺数量或金额的 → drop
- 购方 == 销方 的自交易 → drop（与外包价格语义不符）

In [ ]:
df['year'] = df['开票日期'].dt.year.astype('Int64')
print('year distribution:')
print(df['year'].value_counts(dropna=False).sort_index())

In [ ]:
# 数量缺失反推
m_imp = (df['数量'].isna()
         & df['单价'].notna() & (df['单价'] != 0)
         & df['开票金额'].notna() & df['税额'].notna())
print('quantity imputable rows:', int(m_imp.sum()))
df.loc[m_imp, '数量'] = (df.loc[m_imp, '开票金额'] - df.loc[m_imp, '税额']) / df.loc[m_imp, '单价']

before = len(df)
df = df.dropna(subset=['数量', '开票金额', 'year']).copy()
print(f'dropped (still missing qty/value/year): {before-len(df):,}')

# 税额缺失填 0（某些小规模交易可能不含税）
df['税额'] = df['税额'].fillna(0)

# 自交易
self_mask = (df['购方企业ID'] == df['销方企业ID'])
print('self-transactions:', int(self_mask.sum()), f'({self_mask.mean()*100:.3f}%)')
df = df[~self_mask].copy()

print('rows after Step 5:', len(df))

## Step 6　同 (buyer, seller, product, year) 年度红冲对冲

同一交易对 + 同产品年度内的全部发票（含红冲）加总。净金额或净数量 ≤ 0 丢弃。

购方地区/销方地区 一起进入分组 key，避免同企业偶尔出现不同地区码时被错误合并。

In [ ]:
g = df.groupby(
    ['购方企业ID', '销方企业ID', 'product_id', 'year', '购方地区', '销方地区'],
    as_index=False, dropna=False
).agg(
    value = ('开票金额', 'sum'),
    qty   = ('数量', 'sum'),
    tax   = ('税额', 'sum'),
    n_inv = ('开票金额', 'size'),
)
print('pair-year obs before drop:', len(g))

bad = (g['value'] <= 0) | (g['qty'] <= 0)
print(f'dropped (net value or qty ≤ 0): {int(bad.sum()):,}  ({bad.mean()*100:.2f}%)')
g = g[~bad].copy()

# 送出两版本单价：净价（与原 单价 一致）与含税价
g['p_net']   = (g['value'] - g['tax']) / g['qty']
g['p_gross'] = g['value'] / g['qty']

print('pair-year obs after netting:', len(g))
g.head()

In [ ]:
# 价格分布诊断（可能有极端值）
g[['p_net','p_gross','qty','value']].describe(percentiles=[.01,.05,.5,.95,.99]).round(4)

## Step 7　主回归面板：firm × product × city × year

同一 (购方, 产品, 购方地区, 年) 下，可能从多个卖家购购。合并后加权单价 = 总金额 / 总数量。

In [ ]:
dv = g.groupby(
    ['购方企业ID', 'product_id', '购方地区', 'year'],
    as_index=False
).agg(
    value     = ('value', 'sum'),
    qty       = ('qty',   'sum'),
    tax       = ('tax',   'sum'),
    n_sellers = ('销方企业ID', 'nunique'),
    n_pairs   = ('value', 'size'),
)
dv = dv.rename(columns={'购方企业ID': 'firm_id', '购方地区': 'city'})

dv['p_net']      = (dv['value'] - dv['tax']) / dv['qty']
dv['p_gross']    = dv['value'] / dv['qty']
dv['ln_p_net']   = np.log(dv['p_net'].clip(lower=1e-12))
dv['ln_p_gross'] = np.log(dv['p_gross'].clip(lower=1e-12))
dv['ln_qty']     = np.log(dv['qty'].clip(lower=1e-12))

print('firm × product × city × year obs:', len(dv))
print('unique firms:',    dv['firm_id'].nunique())
print('unique products:', dv['product_id'].nunique())
print('unique cities:',   dv['city'].nunique())
dv.head()

In [ ]:
dv[['p_net','p_gross','qty','n_sellers']].describe(percentiles=[.01,.05,.5,.95,.99]).round(4)

## Step 8　市场条件：product × city(购方地区) × year

以购方所在城市为市场划定（决策 3）：
- `n_buyers`　该城市买该产品的不同购方数 (市场需求宽度)
- `n_sellers`为该城市买家供货的不同销方数（不限销方地区，代表可触及供给）
- `mkt_qty`、`mkt_value`、`mkt_tax`：总金额、总量、总税
- `p_mkt_net`、`p_mkt_gross`：市场均价（各一个版本）

需要时可另外算 sell-side （按 销方地区 划市场）作对比。

In [ ]:
mkt = g.groupby(
    ['product_id', '购方地区', 'year'],
    as_index=False
).agg(
    mkt_value = ('value', 'sum'),
    mkt_qty   = ('qty',   'sum'),
    mkt_tax   = ('tax',   'sum'),
    n_buyers  = ('购方企业ID', 'nunique'),
    n_sellers = ('销方企业ID', 'nunique'),
    n_pairs   = ('value', 'size'),
)
mkt = mkt.rename(columns={'购方地区': 'city'})

mkt['p_mkt_net']    = (mkt['mkt_value'] - mkt['mkt_tax']) / mkt['mkt_qty']
mkt['p_mkt_gross']  = mkt['mkt_value'] / mkt['mkt_qty']
mkt['ln_p_mkt']     = np.log(mkt['p_mkt_net'].clip(lower=1e-12))
mkt['ln_n_buyers']  = np.log(mkt['n_buyers'])
mkt['ln_n_sellers'] = np.log(mkt['n_sellers'])
mkt['ln_mkt_qty']   = np.log(mkt['mkt_qty'].clip(lower=1e-12))

print('product × city × year obs:', len(mkt))
mkt[['n_buyers','n_sellers','mkt_qty','p_mkt_net']].describe(percentiles=[.01,.5,.99]).round(3)

## Step 9　企业特征从 `full_data.dta` 提取

只需要 firm × year 粒度的变量：
- `firm_total_output`　企业总产出（作为规模 proxy）
- `n_products`　　　　企业产品数
- `is_intermediary`　是否为中介企业（>90% 外包）
- `firm_total_outsource`　总外包金额

使用 chunked 读取避免 OOM。

In [ ]:
wanted = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
          'n_products', 'is_intermediary']

itr = pd.read_stata(os.path.join(BASE, 'full_data.dta'),
                    columns=wanted, chunksize=500000)
fc_parts = []
for ch in itr:
    # full_data 是 firm×product×year 面板；firm × year 粒度的变量都是重复值，取首条即可
    ch = ch.drop_duplicates(['firm_id','year'])
    fc_parts.append(ch)
firm_chars = pd.concat(fc_parts, ignore_index=True).drop_duplicates(['firm_id','year'])

firm_chars['firm_id'] = firm_chars['firm_id'].astype(str)
firm_chars['year']    = firm_chars['year'].astype('Int64')
firm_chars['ln_firm_output']    = np.log(firm_chars['firm_total_output'].clip(lower=1))
firm_chars['ln_firm_outsource'] = np.log(firm_chars['firm_total_outsource'].clip(lower=1))

print('firm-year rows:', len(firm_chars))
print('unique firms:',  firm_chars['firm_id'].nunique())
firm_chars.head()

In [ ]:
# 主面板 × 企业特征的 merge 覆盖率诊断
covered = dv['firm_id'].isin(set(firm_chars['firm_id'])).mean()
print(f'DV firms 可以 merge 上 firm_chars 的比例: {covered*100:.2f}%')
print('DV 中 unique firms:', dv['firm_id'].nunique())

## Step 10　导出三个 `.dta`

In [ ]:
# 1) 主回归面板
dv_out = dv[['firm_id','product_id','city','year',
             'value','qty','tax','n_sellers','n_pairs',
             'p_net','p_gross','ln_p_net','ln_p_gross','ln_qty']].copy()
dv_out.to_stata(os.path.join(OUT, 'invoice_panel.dta'),
                write_index=False, version=118)
print('saved invoice_panel.dta  rows =', len(dv_out))

In [ ]:
# 2) 市场条件
mkt_out = mkt[['product_id','city','year',
               'mkt_value','mkt_qty','mkt_tax','n_buyers','n_sellers','n_pairs',
               'p_mkt_net','p_mkt_gross','ln_p_mkt','ln_n_buyers','ln_n_sellers','ln_mkt_qty']].copy()
mkt_out.to_stata(os.path.join(OUT, 'market_conds.dta'),
                 write_index=False, version=118)
print('saved market_conds.dta   rows =', len(mkt_out))

In [ ]:
# 3) 企业特征
firm_out = firm_chars[['firm_id','year','firm_total_output','firm_total_outsource',
                       'n_products','is_intermediary','ln_firm_output','ln_firm_outsource']].copy()
firm_out.to_stata(os.path.join(OUT, 'firm_chars.dta'),
                  write_index=False, version=118)
print('saved firm_chars.dta     rows =', len(firm_out))

In [ ]:
print('=== 清洗完成 ===')
print('all outputs in:', OUT)
for fn in ['invoice_panel.dta', 'market_conds.dta', 'firm_chars.dta']:
    p = os.path.join(OUT, fn)
    sz = os.path.getsize(p) / 1024**2
    print(f'  {fn:25s}  {sz:8.1f} MB')